# 01 · 歌唱デモ（楽譜 → 歌声）

PrettyPitch を F0 モデルとして使い、**楽譜（UST / MusicXML）から歌声を合成**して、
途中の持続長・F0・mel・音声をすべて可視化します。

```
note + lyric
   │
   ├─ ① 歌詞→音素（モーラ表）＋ 子音長モデル（自然長）
   ├─ ② Duration postprocess（floor 保証 / 早口時の比率圧縮 ＋ フレーズ区切り）
   │        └→ phoneme / duration / note pitch
   ├─ ③ F0 モデル：PrettyPitch          → F0 ＋ v/uv
   ├─ ④ F0 postprocess                  → ピッチ補正 / 抑揚 / ビブラート
   ├─ ⑤ 音響モデル：LeapSinger          → mel
   └─ ⑥ ボコーダ：NHVSing v3.2.1          → audio（mel + F0 + v/uv）
```

## 使用モデル

| 役割 | 重み | 備考 |
|---|---|---|
| 子音長 | `checkpoints/cons_dur_3singer.pt` | 3 話者 × 3 スタイル。floor は同名の `duration_model/profiles/cons_dur_3singer.json` |
| F0 | `checkpoints/f0_3singer.pt` | 3 話者 × 3 スタイル・DDPM 10 step |
| 音響 | `3speaker_gan2d.pth` | **v/uv フリー**・3 話者・スタイル無し |
| ボコーダ | `nhv_v3_2_1.onnx` | NHVSing V3.2.1 |

F0・子音長・ボコーダはこのリポに同梱（`checkpoints/`）。音響モデルだけは姉妹リポ
[LeapSinger](https://github.com/wavtechyukky/LeapSinger) の Release の zip を `checkpoints/leapsinger/` に展開し、
LeapSinger のリポジトリを隣（`../LeapSinger`）に clone してください
（`checkpoints/leapsinger/place_leapsinger_models_here.txt` 参照）。

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path

# リポジトリルートを import パスへ（notebooks/ から実行する想定）
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from svs.render import (load_models, render_phrase, render_score,
                        DEFAULT_F0_CKPT, DEFAULT_ACOUSTIC_CKPT, DEFAULT_VOCODER_ONNX)
from svs.score_io import load_score
from svs.duration_postprocess import postprocess_durations, to_frames
from svs.plot import plot_alignment, plot_f0, plot_mel, plot_wave, mel_from_wav

# ウェイトの所在チェック（無ければ何を置くべきか案内する）。
# 表示はリポジトリの親からの相対パス。絶対パスを出力に残さない（notebook の出力は保存されうる）。
for label, p in [('F0', DEFAULT_F0_CKPT), ('acoustic', DEFAULT_ACOUSTIC_CKPT),
                 ('vocoder', DEFAULT_VOCODER_ONNX)]:
    shown = Path(p).relative_to(ROOT.parent) if Path(p).is_relative_to(ROOT.parent) else Path(p).name
    print(f'{label:14s} {"OK " if Path(p).exists() else "!! 見つかりません "} {shown}')

## 1. モデルのロード

`load_models` が 3 モデルをまとめて読み、以降のセルで使い回す `SvsContext` を返します。
`spk_id` は 0=oniku / 1=natsume / 2=ritsu。

In [ ]:
SPK_ID = 2          # 話者名は ctx.speaker_names（子音長 ckpt の spk_map 由来）
SEED = 0

ctx = load_models(spk_id=SPK_ID, device='cpu')
print()
print(f'話者          : {SPK_ID} ({ctx.speaker_names.get(SPK_ID)})')
print(f'use_uv        : {ctx.use_uv}   （False = v/uv フリー経路）')
print(f'frame rate    : {ctx.frame_rate:.3f} fps   ({ctx.sample_rate} Hz / hop {ctx.hop}）')
print(f'音響 n_styles : {ctx.ac_cfg.get("n_styles")}')
print(f'F0   n_styles : {ctx.f0_cfg.get("n_styles")} / n_speakers {ctx.f0_cfg.get("n_speakers")}')

## 2. 楽譜の読み込み（UST / MusicXML どちらでも）

`examples/` には歌詞入りの楽譜が入っています（旋律・歌詞ともパブリックドメイン。
出典は `examples/README.md`）。

| ファイル | 曲 |
|---|---|
| `kirakiraboshi.{ust,musicxml}` | きらきら星（UST と MusicXML は同じ内容） |
| `kaeru_no_uta.musicxml` | かえるのうた |
| `amefuri.musicxml` | あめふり |
| `furusato.musicxml` | ふるさと（3/4） |
| `momiji.musicxml` | もみじ |
| `haru_no_ogawa.musicxml` | 春の小川 |

`load_score` が拡張子で自動判別し、**長い休符でフレーズに分割**します
（既定 `split_silence_sec=0.5`）。以下は「ふるさと」（3 フレーズ・3/4）で進めます。
1 フレーズ目「うさぎおいし かのやま」は伸ばすノートが多く、ビブラートが見やすい題材です。


In [ ]:
SCORE = ROOT / 'examples' / 'furusato.musicxml'      # 既定はふるさと（3 フレーズ）
# 他の曲: kaeru_no_uta / amefuri / momiji / haru_no_ogawa（.musicxml）、kirakiraboshi（.ust / .musicxml）

phrases = load_score(SCORE)
print(f'{SCORE.name}: {len(phrases)} フレーズ / {sum(len(p.notes) for p in phrases)} ノート\n')
for i, p in enumerate(phrases):
    lyric = ''.join(''.join(n.phonemes) for n in p.notes)
    print(f'  フレーズ{i}: {lyric:44s} {len(p.notes):3d} notes  start={p.start_sec:5.2f}s')

# きらきら星だけ UST と MusicXML の両方があるので、2 形式の読み取りが一致することを確認できる
ust, xml = (load_score(ROOT / 'examples' / f'kirakiraboshi.{ext}') for ext in ('ust', 'musicxml'))
same = all(list(postprocess_durations(a, rest_before=1e9).phonemes)
           == list(postprocess_durations(b, rest_before=1e9).phonemes) for a, b in zip(ust, xml))
print(f'\n（きらきら星）UST と MusicXML の音素列が一致: {same}')


## 3. 持続長の生成と後処理

`postprocess_durations` が 1 フレーズを**フラットな音素列 + 各音素の秒数**にします。
子音の自然長は子音長モデル（`checkpoints/cons_dur_3singer.pt`）が出し、後処理がノートに収めます。

* 子音はノートの**前**に置かれる（人はそう歌う）
* **floor 保証**: 母音 floor（75 ms）＋子音 floor（子音別・話者別）がノートに入るなら、
  子音は自然長のまま、母音には floor を残す
* **入らないときだけ**（早口）、子音側 = `ratio × ノート長`（既定 0.5）で圧縮し、母音が残りを取る
* 1 ノートに複数モーラが入っていたら**ノートを等分**し、1 モーラ 1 ノートと同じ扱いにする

このセルは子音長モデルを通さない簡易版（全子音 80 ms のフォールバック）です。モデル込みの
持続長は次の `render_phrase` が計算します。`to_frames` が秒 → フレーム数に変換します（1 フレーム = 256/44100 秒）。


In [ ]:
PH = 0                       # 見るフレーズ
phr = phrases[PH]

seq = postprocess_durations(phr, rest_before=1e9)
ph_durs, T = to_frames(seq.dur_sec, ctx.frame_rate)

print(f'{len(seq.phonemes)} 音素 / {T} フレーム / {seq.dur_sec.sum():.2f} 秒\n')
print(f'{"#":>3} {"ph":<5} {"sec":>6} {"frames":>7} {"midi":>5} {"vowel":>6} {"note_start":>11}')
for i, ph in enumerate(seq.phonemes[:20]):
    print(f'{i:>3} {ph:<5} {seq.dur_sec[i]:6.3f} {ph_durs[i]:7d} {seq.note_midi[i]:5d} '
          f'{str(bool(seq.is_vowel[i])):>6} {str(bool(seq.note_event_starts[i])):>11}')

fig, ax = plt.subplots(figsize=(14, 2.4))
plot_alignment(seq, ax=ax); plt.tight_layout(); plt.show()

## 4. 合成（F0 → mel → 波形）

`render_phrase` が ③〜⑥ をまとめて実行し、途中経過ごと返します。

In [ ]:
r = render_phrase(ctx, phr, seed=SEED)
seq, f0o, mel, wav = r['seq'], r['f0'], r['mel'], r['wav']

voiced = f0o['uv'] > 0
print(f"frames {len(f0o['f0_hz'])} | mel {mel.shape} | wav {len(wav)/ctx.sample_rate:.2f}s")
print(f"F0  {f0o['f0_hz'][voiced].min():.1f}–{f0o['f0_hz'][voiced].max():.1f} Hz  "
      f"(有声率 {voiced.mean()*100:.0f}%)")
print(f"mel {mel.min():.2f}..{mel.max():.2f}")

### 4-1. F0

破線 = `baseline`（楽譜の音高を階段状にしてガウシアン平滑したもの）、
実線 = 生成 F0（2 段ピッチ補正まで適用済み）。
右軸は `plain`（抑揚）と `vibrato` の寄与を cent で表示しています。
**この 2 つは独立に強度を変えられます**（→ notebook 03）。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), gridspec_kw={'height_ratios': [1, 2.4]})
plot_alignment(seq, ax=axes[0])
plot_f0(f0o, ax=axes[1], seq=seq)
axes[1].set_xlim(axes[0].get_xlim())
plt.tight_layout(); plt.show()

### 4-2. mel（音響モデルの出力）

音響モデルは **v/uv フリー**です（v/uv を条件に取る版は LeapSinger 側にあり、`--acoustic_ckpt` で差せます）。話者は `spk_bank` の離散 lookup と同値のベクトルを
`spk_embed` として渡しています（これを省くと 3 話者モデルでも話者条件が効きません）。

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
plot_mel(mel, ctx.frame_rate, ax=ax, title=f'predicted mel (uv-free, spk={ctx.speaker_names.get(SPK_ID)})')
plt.tight_layout(); plt.show()

### 4-3. 波形と音声（ボコーダ NHVSing v3.2.1）

ボコーダには mel・F0 に加えて **v/uv（F0 モデルの vuv ∧ 非 pau）** を渡しています。


In [ ]:
fig, ax = plt.subplots(figsize=(14, 2))
plot_wave(wav, ctx.sample_rate, ax=ax); plt.tight_layout(); plt.show()
display(Audio(wav, rate=ctx.sample_rate))

## 5. 出力音声を再分析した mel

ボコーダは「音響モデルが要求した mel」をどれだけ再現できているか。
`mel_from_wav` はボコーダ学習時と**厳密に同じレシピ**（`center=False` ＋ reflect pad、
hann、自然対数）なので、予測 mel とフレーム単位で揃い、そのまま差分が取れます。

ただし mel の段でボコーダ向けに音量を合わせてあるので、再分析 mel は全体が一定量
（この例で約 +6 dB）持ち上がります。それは音量の差であって形の差ではないので、
有声フレームの差の中央値を引いてから比べます。

mel に掛かるのは 2 つだけです。ボコーダを学習時と同じ音量で駆動するための定数の足し算と、
熱すぎるフレームを下げるリミッタです（[guide/svs_pipeline.md](../guide/svs_pipeline.md) の
「音量を整える」）。クランプもスムージングも掛けません。その証拠に `mel.min()` は config の
`mel_vmin`（-11.5）を下回ります。


In [ ]:
remel = mel_from_wav(wav, sr=ctx.sample_rate, hop=ctx.hop)
T = min(remel.shape[1], mel.shape[1])
diff = remel[:, :T] - mel[:, :T]
voiced_T = f0o['uv'][:T] > 0.5
gain = float(np.median(diff[:, voiced_T]))          # mel 段の音量合わせによるレベル差（ln）
diff = diff - gain
mae = float(np.abs(diff).mean()); mae_v = float(np.abs(diff[:, voiced_T]).mean())
print(f'予測 mel {mel.shape} / 再分析 mel {remel.shape}')
print(f'レベル差 {gain:+.2f} ln ({gain*20/np.log(10):+.1f} dB) を除いた ln-mel MAE = {mae:.3f}（有声 {mae_v:.3f}）')
print(f"mel.min() = {mel.min():.2f}  vs  mel_vmin = {ctx.ac_cfg.get('mel_vmin')} "
      f"→ クランプされていない")

fig, axes = plt.subplots(3, 1, figsize=(14, 8))
plot_mel(mel, ctx.frame_rate, ax=axes[0], title='predicted mel (acoustic model)')
plot_mel(remel[:, :T] - gain, ctx.frame_rate, ax=axes[1],
         title=f're-analyzed mel from output audio, level-aligned (MAE={mae:.3f})')
plot_mel(diff, ctx.frame_rate, ax=axes[2], vmin=-2, vmax=2,
         title='difference (re-analyzed − predicted, level-aligned)')
plt.tight_layout(); plt.show()


## 6. 曲全体

`render_score` が全フレーズを合成し、楽譜の絶対時刻でタイムラインに並べます
（配置は `start_sec − 先頭pau − 先頭onset` = 音節核が `start_sec` に来る位置。
負になるフレーズがあれば曲全体を後ろへずらす。重なりは単純加算。音量は mel の段で
決まっているので、波形には手を入れません）。

In [ ]:
wav_all, sr = render_score(SCORE, ctx=ctx, seed=SEED)
print(f'\n{len(wav_all)/sr:.2f} 秒')
display(Audio(wav_all, rate=sr))

# 保存したいとき:
# import soundfile as sf; sf.write('out/furusato.wav', wav_all, sr)

## 7. 話者を変える

3 モデルとも話者を持っています（子音長・F0 は `spk_id`、音響は同じ id の話者ベクトル）。
`SvsContext` は話者 id と音響用の話者ベクトルを束ねているだけなので、`dataclasses.replace`
で差し替えれば重みを読み直さずに話者を切り替えられます。同じフレーズ・同じシードで 3 話者を並べます。

In [ ]:
import dataclasses
from svs.render import _speaker_vector

by_spk = {}
for sid, name in sorted(ctx.speaker_names.items()):
    ctx_s = dataclasses.replace(ctx, spk_id=sid, spk_vec=_speaker_vector(ctx.ac_model, sid, ctx.device))
    rs = render_phrase(ctx_s, phr, seed=SEED)
    by_spk[sid] = (name, rs)
    s = rs['seq']; cons = s.dur_sec[(~s.is_vowel) & (np.asarray(s.phonemes) != 'pau')]
    o = rs['f0']; v = o['uv'] > 0
    print(f'spk {sid} {name:8s} 子音平均 {cons.mean()*1000:5.1f}ms  有声率 {v.mean()*100:.0f}%  '
          f'plain σ {o["plain_delta"][v].std()*1200:5.1f}c  vibrato σ {o["vib_delta"][v].std()*1200:5.1f}c')

fig, axes = plt.subplots(2, 1, figsize=(14, 7.5), sharex=True, gridspec_kw={'height_ratios': [2, 1.1]})
colors = ['#6a3d9a', '#1565c0', '#e6550d']
for (sid, (name, rs)), col in zip(by_spk.items(), colors):
    o = rs['f0']; t = np.arange(len(o['f0_hz'])) / float(o['frame_rate'])
    midi = np.where(o['uv'] > 0, 69 + 12 * np.log2(np.maximum(o['f0_hz'], 1e-6) / 440.0), np.nan)
    axes[0].plot(t, midi, lw=1.2, color=col, label=f'{sid}: {name}')
    axes[1].plot(t, o['plain_delta'] * 1200, lw=0.9, color=col)
    axes[1].plot(t, o['vib_delta'] * 1200, lw=0.9, ls=':', color=col)
o0 = f0o; t0 = np.arange(len(o0['f0_hz'])) / float(o0['frame_rate'])
axes[0].plot(t0, 69 + 12 * (o0['baseline_log2'] - np.log2(440.0)), '--', lw=1.0, color='#888', label='baseline')
axes[0].set_ylabel('pitch [MIDI]'); axes[0].legend(fontsize=8, loc='upper right')
axes[0].set_title(f'3 speakers, same phrase, seed={SEED}')
axes[1].axhline(0, color='#888', lw=0.5); axes[1].set_ylabel('deviation [cent]'); axes[1].set_xlabel('time [s]')
axes[1].set_title('plain = solid, vibrato = dotted', fontsize=10)
plt.tight_layout(); plt.show()

for sid, (name, rs) in by_spk.items():
    print(f'spk {sid}: {name}'); display(Audio(rs['wav'], rate=ctx.sample_rate))


## 8. スタイルの連続変化（ritsu → ritsu soft）

F0 モデルと子音長モデルはどちらもスタイルを**埋め込みの加算**で受け取っています
（0 = ritsu（既定）/ 1 = ritsu normal / 2 = ritsu soft）。整数の `style_id` の代わりに
`style_mix=[w0, w1, w2]` を渡すと、埋め込み表をその比で混ぜた 1 本のベクトルが加算されます
（one-hot なら `style_id` と同じ結果）。ここでは ritsu（α=0）から ritsu soft（α=1）まで
0.2 刻みで動かし、**同一シード**で F0 と子音長がどう変わるかを見ます。

* 動くもの: F0 モデル（plain / vibrato / v-uv）と子音長モデルの自然長
* 動かないもの: floor プロファイル（`style_id` のまま）

音響モデルにはスタイルがない（`n_styles=0`）ので、ここでは音声は作らず、`render_phrase(..., f0_only=True)` で
持続長と F0 だけを出してプロットします（題材は引き続き 1 フレーズ目 `phr`）。

In [ ]:
STYLE_FROM, STYLE_TO = 0, 2                        # ritsu → ritsu soft
ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
N_STYLES = int(ctx.f0_cfg['n_styles'])

def style_mix(alpha):
    w = np.zeros(N_STYLES, np.float32)
    w[STYLE_FROM] = 1.0 - alpha; w[STYLE_TO] = alpha
    return w

morph = []
for a in ALPHAS:
    rm = render_phrase(ctx, phr, seed=SEED, style_id=STYLE_FROM, style_mix=style_mix(a),
                       f0_only=True)
    s = rm['seq']
    cons = s.dur_sec[(~s.is_vowel) & (np.asarray(s.phonemes) != 'pau')]
    f0o = rm['f0']; v = f0o['uv'] > 0
    morph.append((a, rm))
    print(f'α={a:.1f}  全長 {s.dur_sec.sum():.2f}s  子音平均 {cons.mean()*1000:5.1f}ms  '
          f'有声率 {v.mean()*100:.0f}%  plain σ {f0o["plain_delta"][v].std()*1200:5.1f}c  '
          f'vibrato σ {f0o["vib_delta"][v].std()*1200:5.1f}c')


上段: 生成 F0（MIDI）。破線は α=0 の baseline（楽譜）。下段: baseline からのずれ
（実線 = plain（抑揚）、点線 = vibrato）。色が濃い（紫）ほど ritsu、明るい（黄）ほど ritsu soft。
子音長も一緒に変わるので、後ろのノートほど時間軸が少しずれます。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7.5), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1.1]})
cmap = plt.get_cmap('viridis')
for a, rm in morph:
    o = rm['f0']; t = np.arange(len(o['f0_hz'])) / float(o['frame_rate'])
    midi = np.where(o['uv'] > 0, 69 + 12 * np.log2(np.maximum(o['f0_hz'], 1e-6) / 440.0), np.nan)
    axes[0].plot(t, midi, lw=1.2, color=cmap(a), label=f'α={a:.1f}')
    axes[1].plot(t, o['plain_delta'] * 1200, lw=0.9, color=cmap(a))
    axes[1].plot(t, o['vib_delta'] * 1200, lw=0.9, ls=':', color=cmap(a))
o0 = morph[0][1]['f0']; t0 = np.arange(len(o0['f0_hz'])) / float(o0['frame_rate'])
axes[0].plot(t0, 69 + 12 * (o0['baseline_log2'] - np.log2(440.0)), '--', lw=1.0, color='#888',
             label='baseline (α=0)')
axes[0].set_ylabel('pitch [MIDI]'); axes[0].legend(ncol=4, fontsize=8, loc='upper right')
axes[0].set_title(f'style morph: ritsu (α=0) → ritsu soft (α=1), seed={SEED}')
axes[1].axhline(0, color='#888', lw=0.5)
axes[1].set_ylabel('deviation [cent]'); axes[1].set_xlabel('time [s]')
axes[1].set_title('plain = solid, vibrato = dotted', fontsize=10)
plt.tight_layout(); plt.show()


## 9. 抑揚とビブラートの強さ

`synthesize_f0` は F0 を **baseline（楽譜）＋ plain（抑揚）＋ vibrato** に分けて持っていて、
`plain_scale` / `vib_scale` はそれぞれの偏差に掛かる倍率です（既定 1.0）。
モデルの出力そのものは変わらず、同一シードならノイズも同じなので、**振幅だけが変わります**。
plain の倍率はピッチ補正（2 段）の前に掛かるので、補正後の見た目はちょうど倍率分にはなりません。
子音長は F0 モデルの外なので変わりません。

まず 0.5 / 1.0 / 1.5 の 3 × 3 = 9 通りを同じフレーズ・同じシードで合成します。

In [ ]:
SCALES = [0.5, 1.0, 1.5]
grid = {}
for ps_ in SCALES:
    for vs_ in SCALES:
        grid[(ps_, vs_)] = render_phrase(ctx, phr, seed=SEED, plain_scale=ps_, vib_scale=vs_)

print(f'{"plain / vib":>12}' + ''.join(f'{"vib " + str(vs_):>26s}' for vs_ in SCALES))
for ps_ in SCALES:
    row = []
    for vs_ in SCALES:
        o = grid[(ps_, vs_)]['f0']; v = o['uv'] > 0
        row.append(f'plain σ {o["plain_delta"][v].std()*1200:3.0f}c / vib σ {o["vib_delta"][v].std()*1200:2.0f}c')
    print(f'{"plain " + str(ps_):>12s}' + ''.join(f'{c:>26s}' for c in row))


左列: 生成 F0（MIDI）。右列: baseline からのずれ [cent]。上段は `plain_scale` を変えたとき
（`vib_scale=1`）、下段は `vib_scale` を変えたとき（`plain_scale=1`）。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 7.5), sharex=True,
                         gridspec_kw={'width_ratios': [1.6, 1]})
colors = {0.5: '#1b9e77', 1.0: '#1565c0', 1.5: '#d62728'}
base_o = grid[(1.0, 1.0)]['f0']; tb = np.arange(len(base_o['f0_hz'])) / float(base_o['frame_rate'])
for row, (knob, key) in enumerate((('plain_scale', 'plain_delta'), ('vib_scale', 'vib_delta'))):
    for s_ in SCALES:
        o = grid[(s_, 1.0) if knob == 'plain_scale' else (1.0, s_)]['f0']
        t = np.arange(len(o['f0_hz'])) / float(o['frame_rate'])
        midi = np.where(o['uv'] > 0, 69 + 12 * np.log2(np.maximum(o['f0_hz'], 1e-6) / 440.0), np.nan)
        axes[row, 0].plot(t, midi, lw=1.1, color=colors[s_], label=f'{knob}={s_}')
        axes[row, 1].plot(t, o[key] * 1200, lw=0.9, color=colors[s_], label=f'{knob}={s_}')
    axes[row, 0].plot(tb, 69 + 12 * (base_o['baseline_log2'] - np.log2(440.0)), '--', lw=1.0,
                      color='#888', label='baseline')
    axes[row, 0].set_ylabel('pitch [MIDI]'); axes[row, 0].legend(fontsize=8, loc='upper right')
    axes[row, 1].axhline(0, color='#888', lw=0.5); axes[row, 1].set_ylabel(f'{key} [cent]')
    axes[row, 1].legend(fontsize=8, loc='upper right')
axes[0, 0].set_title('F0 (plain_scale varies, vib_scale=1)'); axes[0, 1].set_title('plain deviation')
axes[1, 0].set_title('F0 (vib_scale varies, plain_scale=1)');  axes[1, 1].set_title('vibrato deviation')
for ax in axes[1]: ax.set_xlabel('time [s]')
plt.tight_layout(); plt.show()


9 通りの音声。行が `plain_scale`（抑揚）、列が `vib_scale`（ビブラート）。

In [ ]:
from IPython.display import HTML

def _audio_html(wav, sr):
    return Audio(wav, rate=sr)._repr_html_().replace('<audio', '<audio style="width:250px"')

rows = ['<tr><th></th>' + ''.join(f'<th>vib_scale = {vs_}</th>' for vs_ in SCALES) + '</tr>']
for ps_ in SCALES:
    rows.append(f'<tr><th>plain_scale = {ps_}</th>'
                + ''.join(f'<td>{_audio_html(grid[(ps_, vs_)]["wav"], ctx.sample_rate)}</td>' for vs_ in SCALES)
                + '</tr>')
display(HTML('<table>' + ''.join(rows) + '</table>'))


---

## 次に読むもの

* **02_design_notes.ipynb** — 持続長 / F0 / mel にどんな補正が掛かっていて、なぜそうしているか
* **03_f0_control.ipynb** — 抑揚とビブラートを独立に、さらに**フレーズの中で時間変化**させる方法